In [1]:
import pandas as pd
import pickle

In [2]:
# ---- 1. Chargement des données ----

# Fichier contenant les phrases segmentées
df_phrases = pd.read_csv("./artifacts/bertopic/reviews_phrases_with_topics.csv")

# Fichier clusters annotés
df_clusters = pd.read_excel("./artifacts/bertopic/topics_top_words_phrases_annoté.xlsx")

replace_map = {
    # qualité produit
    "Qualité Produit": "qualité produit",
    "Qualité produit": "qualité produit",

    # livraison
    "Service Livraison": "service livraison",
    "Service livraison": "service livraison",

    # service client
    "Service Client": "service client",
    "Service client": "service client",
}

def normalize_category(cat):
    if pd.isna(cat):
        return None
    cat = cat.strip().lower()
    return replace_map.get(cat, cat)


df_clusters["Catégorie"] = df_clusters["Catégorie"].apply(normalize_category)

# ---- 2. Préparation du fichier des phrases ----

# Fusion sur le numéro de topic
df_phrases_merged = df_phrases.merge(
    df_clusters[["Topic", "Catégorie"]],
    left_on='topics',
    right_on='Topic',
    how='left'
).drop(columns=['Topic'])

# Catégories possibles
categories = ["qualité produit", "service livraison", "service client"]

# Retirer les neutres si nécessaire
df_phrases_merged = df_phrases_merged[df_phrases_merged['Catégorie'].isin(categories)]


# Création colonnes one-hot binaires
for cat in categories:
    df_phrases_merged[cat] = (df_phrases_merged['Catégorie'] == cat).astype(int)

df_phrases.to_csv("./../resultats/bertopic/data/dataset_phrases.csv", index=False)

In [3]:
cols_invariantes = ["Commentaire", "star", "date", "client", "reponse", "source", "company", "ville", "maj", "date_commande", "ecart", "clean_comment"]

agg_dict = {}

# Colonnes invariantes → first
for col in cols_invariantes:
    agg_dict[col] = (col, "first")

# Colonnes de labels → max
for col in categories:
    agg_dict[col] = (col, "max")

df_avis = (
    df_phrases_merged
    .groupby("comment_id", as_index=False)
    .agg(**agg_dict)
)

df_avis.to_csv("./../resultats/bertopic/data/dataset_avis.csv", index=False)

In [4]:
df_phrases_merged

,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,comment_id,sentence,topics,Catégorie,qualité produit,service livraison,service client
16,"Vente Lacoste Honteuse , article erroné , arti...",1,2021-06-19 00:00:00+00:00,Vanessa L,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"vente lacoste honteuse , article erroné , arti...",2,( commande n°230077467 et commande n•230077467...,97,qualité produit,1,0,0
24,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,annulation de commande après 2 mois d ’ attent...,6,annulation de commande après 2 mois d ’ attent...,62,service client,0,0,1
32,Extrêmement deçu pour mes achats lors la vente...,1,2021-06-18 00:00:00+00:00,Ayna,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,extrêmement deçu pour mes achats lors la vente...,8,extrêmement deçu pour mes achats lors la vente...,87,qualité produit,1,0,0
33,S'il y'avait une option : ne pas mettre d'étoi...,1,2021-06-18 00:00:00+00:00,linda Ng,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,s'il y'avait une option : ne pas mettre d'étoi...,9,s'il y'avait une option : ne pas mettre d'étoi...,134,qualité produit,1,0,0
40,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,arnaque j ’ ai acheté une combinaison blanche ...,10,grosse arnaque c ’ est inadmissible je vais la...,87,qualité produit,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37600,J'ai passé 4 fois commande chez eux ces derniè...,3,2015-10-15 00:00:00+00:00,Client,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,j'ai passé 4 fois commande chez eux ces derniè...,15071,livraison prévue entre le 18 et le 2 octobre (...,189,service livraison,0,1,0
37610,Cliente depuis 2008 sans encombre jusqu ' à ju...,1,2015-10-06 00:00:00+00:00,Bertho,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,cliente depuis 2008 sans encombre jusqu ' à ju...,15073,les réponses toutes faites : rupture de stock !,88,qualité produit,1,0,0
37623,Je suis client sur ce site depuis plusieurs an...,5,2015-10-02 00:00:00+00:00,Thomas GUILLAUME,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,je suis client sur ce site depuis plusieurs an...,15075,les marques sont variés et concernent des prod...,145,qualité produit,1,0,0
37624,Je suis client sur ce site depuis plusieurs an...,5,2015-10-02 00:00:00+00:00,Thomas GUILLAUME,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,je suis client sur ce site depuis plusieurs an...,15075,les prix sont parfois aléatoires ( de la tres ...,61,qualité produit,1,0,0


In [5]:
df_avis

,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,2,"Vente Lacoste Honteuse , article erroné , arti...",1,2021-06-19 00:00:00+00:00,Vanessa L,None,TrustPilot,ShowRoom,None,None,None,NaN,"vente lacoste honteuse , article erroné , arti...",1,0,0
1,6,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,None,None,None,NaN,annulation de commande après 2 mois d ’ attent...,0,0,1
2,8,Extrêmement deçu pour mes achats lors la vente...,1,2021-06-18 00:00:00+00:00,Ayna,None,TrustPilot,ShowRoom,None,None,None,NaN,extrêmement deçu pour mes achats lors la vente...,1,0,0
3,9,S'il y'avait une option : ne pas mettre d'étoi...,1,2021-06-18 00:00:00+00:00,linda Ng,None,TrustPilot,ShowRoom,None,None,None,NaN,s'il y'avait une option : ne pas mettre d'étoi...,1,0,0
4,10,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,None,TrustPilot,ShowRoom,None,None,None,NaN,arnaque j ’ ai acheté une combinaison blanche ...,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6530,15065,Je suis une cliente depuis 10 ans de Vente pri...,4,2015-10-21 00:00:00+00:00,Claudie,None,TrustPilot,VeePee,None,None,None,NaN,je suis une cliente depuis 10 ans de vente pri...,1,0,0
6531,15070,J'ai voulu commandé sur le site plusieurs arti...,1,2015-10-15 00:00:00+00:00,Mr stephane D .,None,TrustPilot,VeePee,None,None,None,NaN,j'ai voulu commandé sur le site plusieurs arti...,1,1,0
6532,15071,J'ai passé 4 fois commande chez eux ces derniè...,3,2015-10-15 00:00:00+00:00,Client,None,TrustPilot,VeePee,None,None,None,NaN,j'ai passé 4 fois commande chez eux ces derniè...,0,1,0
6533,15073,Cliente depuis 2008 sans encombre jusqu ' à ju...,1,2015-10-06 00:00:00+00:00,Bertho,None,TrustPilot,VeePee,None,None,None,NaN,cliente depuis 2008 sans encombre jusqu ' à ju...,1,0,0


In [19]:
embedding_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "camembert-base",
    "paraphrase-multilingual-MiniLM-L12-v2"
]

max_depths = [3, 4, 5]

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [21]:
f1_micro_scores = []
f1_weighted_scores = []

for emb_name in embedding_models:
    print(f"\nEmbedding : {emb_name}")
    embedder = SentenceTransformer(emb_name)

    for max_depth in max_depths:
        print(f"max_depth = {max_depth}")

        f1_micro_scores = []
        f1_weighted_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(texts), 1):

            # Split
            X_train_texts = texts[train_idx]
            X_val_texts = texts[val_idx]
            y_train = labels[train_idx]
            y_val = labels[val_idx]

            # Embeddings
            X_train = embedder.encode(X_train_texts, convert_to_numpy=True, show_progress_bar=False)
            X_val = embedder.encode(X_val_texts, convert_to_numpy=True, show_progress_bar=False)

            # Modèle
            model = MultiOutputClassifier(
                XGBClassifier(
                    eval_metric="logloss",
                    n_estimators=100,
                    max_depth=max_depth,
                    learning_rate=0.1,
                    random_state=42
                )
            )

            # Entraînement
            model.fit(X_train, y_train)

            # Prédiction
            y_pred = model.predict(X_val)

            # Scores
            f1_micro_scores.append(
                f1_score(y_val, y_pred, average="micro")
            )
            f1_weighted_scores.append(
                f1_score(y_val, y_pred, average="weighted")
            )

        print(
            f"    F1 micro: {np.mean(f1_micro_scores):.4f} | "
            f"F1 weighted: {np.mean(f1_weighted_scores):.4f}"
        )



Embedding : all-MiniLM-L6-v2
max_depth = 3
    F1 micro: 0.6955 | F1 weighted: 0.6734
max_depth = 4
    F1 micro: 0.6993 | F1 weighted: 0.6797
max_depth = 5
    F1 micro: 0.7031 | F1 weighted: 0.6825

Embedding : all-mpnet-base-v2
max_depth = 3
    F1 micro: 0.6914 | F1 weighted: 0.6704
max_depth = 4
    F1 micro: 0.6976 | F1 weighted: 0.6787
max_depth = 5
    F1 micro: 0.6984 | F1 weighted: 0.6779

Embedding : camembert-base


No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


max_depth = 3
    F1 micro: 0.7348 | F1 weighted: 0.7214
max_depth = 4
    F1 micro: 0.7367 | F1 weighted: 0.7237
max_depth = 5
    F1 micro: 0.7369 | F1 weighted: 0.7234

Embedding : paraphrase-multilingual-MiniLM-L12-v2
max_depth = 3
    F1 micro: 0.7022 | F1 weighted: 0.6857
max_depth = 4
    F1 micro: 0.7083 | F1 weighted: 0.6923
max_depth = 5
    F1 micro: 0.7060 | F1 weighted: 0.6885


In [22]:
embedder = SentenceTransformer('camembert-base')
max_depth = 5

X_train = embedder.encode(texts, convert_to_numpy=True)
X_test = embedder.encode(X_test, convert_to_numpy=True)

No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


In [23]:
# Modèle
model = MultiOutputClassifier(
    XGBClassifier(
        eval_metric="logloss",
        n_estimators=100,
        max_depth=max_depth,
        learning_rate=0.1,
        random_state=42
    )
)

# Entraînement
model.fit(X_train, labels)

# Prédictions
y_pred = model.predict(X_test)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"F1 micro     : {f1_micro:.4f}")
print(f"F1 weighted  : {f1_weighted:.4f}")

Label 'qualité produit': Accuracy = 0.786, F1-score = 0.816
Label 'service livraison': Accuracy = 0.793, F1-score = 0.731
Label 'service client': Accuracy = 0.866, F1-score = 0.403
F1 micro     : 0.7430
F1 weighted  : 0.7260


## CamemBERT fine-tuné

In [1]:
import torch
import transformers
import datasets
import sklearn
import pandas as pd
from transformers import CamembertForSequenceClassification, CamembertTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import f1_score

/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_avis = pd.read_csv('./../resultats/bertopic/data/dataset_avis.csv')

In [3]:
TEXT_COL = "clean_comment"
LABEL_COLS = ["qualité produit", "service livraison", "service client"]

label_names = LABEL_COLS
num_labels = len(label_names)

print(label_names)

['qualité produit', 'service livraison', 'service client']


In [4]:
from sklearn.model_selection import KFold, train_test_split
import numpy as np

X = df_avis[TEXT_COL].values
y = df_avis[LABEL_COLS].values

texts, X_test, labels, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
df_avis_gold = pd.read_csv('./../data/test_dataset/100_avis_annote.csv', sep=";")
X_gold = df_avis_gold[TEXT_COL].values
y_gold = df_avis_gold[LABEL_COLS].values

## Entraînement avec cross validation

#### Sélection des hyperparamètres

In [96]:
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

In [97]:
from transformers import CamembertForSequenceClassification, CamembertTokenizer
from transformers import Trainer, TrainingArguments
import torch
from sklearn.metrics import f1_score
import numpy as np
import torch
from torch.utils.data import Dataset

all_fold_metrics = []

for fold, (train_index, val_index) in enumerate(kf.split(texts)):
    print(f"\nFold {fold + 1}/{k}")

    # Split train/val
    train_texts, val_texts = texts[train_index], texts[val_index]
    train_labels, val_labels = labels[train_index], labels[val_index]

    # Tokenizer
    tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
    
    # Créer des datasets Hugging Face
    def tokenize_and_format(texts, labels):
        encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=256)
        encodings['labels'] = labels.astype(np.float32)
        return encodings

    train_enc = tokenize_and_format(train_texts, train_labels)
    val_enc = tokenize_and_format(val_texts, val_labels)

    class MyDataset(Dataset):
        def __init__(self, encodings):
            self.encodings = encodings
        def __len__(self):
            return len(self.encodings['input_ids'])
        def __getitem__(self, idx):
            return {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}

    train_ds = MyDataset(train_enc)
    val_ds = MyDataset(val_enc)

    # Modèle
    model = CamembertForSequenceClassification.from_pretrained(
        "camembert-base",
        num_labels=len(LABEL_COLS),
        problem_type="multi_label_classification"
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Trainer
    training_args = TrainingArguments(
        output_dir=f"./../models/camembert/camembert_fold{fold}",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_micro",
        fp16=True  # accélération GPU
    )

    # Fonction métriques
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.sigmoid(torch.tensor(logits))
        preds = (probs > 0.5).int().numpy()
        labels = labels.astype(int)
        f1_micro = f1_score(labels, preds, average="micro")
        return {"f1_micro": f1_micro}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Entraînement
    trainer.train()

    # Évaluation finale sur ce fold
    metrics = trainer.evaluate()
    print(metrics)
    all_fold_metrics.append(metrics['eval_f1_micro'])

# Moyenne sur les folds
print("F1 micro moyen sur les folds:", np.mean(all_fold_metrics))



Fold 1/5


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_619/2843108505.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro
1,0.466800,0.454648,0.713111
2,0.414700,0.426176,0.736706
3,0.398600,0.415033,0.753661


{'eval_loss': 0.4150334894657135, 'eval_f1_micro': 0.7536606373815676, 'eval_runtime': 3.9468, 'eval_samples_per_second': 265.023, 'eval_steps_per_second': 16.722, 'epoch': 3.0}

Fold 2/5


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_619/2843108505.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Epoch,Training Loss,Validation Loss,F1 Micro
1,0.454300,0.458916,0.714798
2,0.413000,0.444950,0.730159
3,0.397000,0.435112,0.740868


{'eval_loss': 0.4351119101047516, 'eval_f1_micro': 0.740868070477009, 'eval_runtime': 3.8998, 'eval_samples_per_second': 267.961, 'eval_steps_per_second': 16.924, 'epoch': 3.0}

Fold 5/5


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_619/2843108505.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro
1,0.483900,0.444318,0.731816
2,0.403900,0.426484,0.742074
3,0.371800,0.423325,0.751405


{'eval_loss': 0.4233245849609375, 'eval_f1_micro': 0.7514051015996541, 'eval_runtime': 4.048, 'eval_samples_per_second': 258.151, 'eval_steps_per_second': 16.304, 'epoch': 3.0}
F1 micro moyen sur les folds: 0.7498064048926968


#### Entraînement final

In [5]:
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

def tokenize_and_format(texts, labels):
        encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=256)
        encodings['labels'] = labels.astype(np.float32)
        return encodings

train_enc = tokenize_and_format(texts, labels)

In [6]:
class MyDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items() if k != 'labels'}
        item['labels'] = torch.tensor(self.encodings['labels'][idx], dtype=torch.float)
        return item

train_ds = MyDataset(train_enc)

In [7]:
model_final = CamembertForSequenceClassification.from_pretrained(
    "camembert-base",
    num_labels=len(LABEL_COLS),
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_final.to(device)

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CamembertForSequenceClassification(
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()  # ajuster les seuils optimaux
    labels = labels.astype(int)
    f1_micro = f1_score(labels, preds, average="micro")
    return {"f1_micro": f1_micro}

In [9]:
training_args_final = TrainingArguments(
    output_dir="./camembert_final",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2, 
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    fp16=True
)

In [10]:
trainer_final = Trainer(
    model=model_final,
    args=training_args_final,
    train_dataset=train_ds,
    compute_metrics=compute_metrics
)

In [11]:
trainer_final.train()

Step,Training Loss
100,0.606500
200,0.504700
300,0.455500
400,0.441000
500,0.410800
600,0.417400


TrainOutput(global_step=654, training_loss=0.46780218480194746, metrics={'train_runtime': 138.9297, 'train_samples_per_second': 75.261, 'train_steps_per_second': 4.707, 'total_flos': 1375556947881984.0, 'train_loss': 0.46780218480194746, 'epoch': 2.0})

In [104]:
trainer_final.save_model("./../models/camembert/camembert_final")
tokenizer.save_pretrained("./../models/camembert/camembert_final")
print("Modèle final entraîné et sauvegardé ✅")

Modèle final entraîné et sauvegardé ✅


#### Evaluation sur le dataset de test

In [12]:
# Tokenisation
test_enc = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)
test_enc['labels'] = torch.tensor(y_test, dtype=torch.float)

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: v[idx] for k,v in self.encodings.items()}

test_ds = TestDataset(test_enc)

# Evaluation
trainer_final.eval_dataset = test_ds
metrics = trainer_final.evaluate()
print(metrics)

{'eval_loss': 0.411691278219223, 'eval_f1_micro': 0.7585967349774227, 'eval_runtime': 3.4172, 'eval_samples_per_second': 382.473, 'eval_steps_per_second': 23.996, 'epoch': 2.0}


In [13]:
import torch
from sklearn.metrics import f1_score

texts = list(X_test)
labels_true = torch.tensor(y_test, dtype=torch.int)

# Préparer les inputs
inputs = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
inputs = {k: v.to(model_final.device) for k,v in inputs.items()}

# Prédictions
with torch.no_grad():
    logits = model_final(**inputs).logits
probs = torch.sigmoid(logits).cpu()

# Appliquer les seuils pour chaque label
thresholds = [0.5, 0.5, 0.5]
preds = (probs > torch.tensor(thresholds)).int()

# F1 score
f1_micro = f1_score(labels_true, preds, average='micro')
f1_macro = f1_score(labels_true, preds, average='macro')

print("F1 micro:", f1_micro)
print("F1 macro:", f1_macro)

F1 micro: 0.7585967349774227
F1 macro: 0.694424779848159


In [14]:
from sklearn.metrics import classification_report

print(classification_report(labels_true, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.81      0.80      0.81       717
service livraison       0.77      0.79      0.78       547
   service client       0.62      0.42      0.50       209

        micro avg       0.78      0.74      0.76      1473
        macro avg       0.73      0.67      0.69      1473
     weighted avg       0.77      0.74      0.75      1473
      samples avg       0.79      0.77      0.77      1473



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [15]:
from sklearn.metrics import confusion_matrix

for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(labels_true[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,458,132
Vrai 1,143,574


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,632,128
Vrai 1,116,431


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,1044,54
Vrai 1,122,87


#### Evaluation sur le dataset gold annoté manuellement

In [19]:
# Tokenisation
gold_enc = tokenizer(
    list(X_gold),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)
gold_enc['labels'] = torch.tensor(y_gold, dtype=torch.float)

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: v[idx] for k,v in self.encodings.items()}

gold_ds = TestDataset(gold_enc)

# Evaluation
trainer_final.eval_dataset = gold_ds
metrics = trainer_final.evaluate()
print(metrics)

{'eval_loss': 0.5943608283996582, 'eval_f1_micro': 0.5851063829787234, 'eval_runtime': 0.4408, 'eval_samples_per_second': 226.854, 'eval_steps_per_second': 15.88, 'epoch': 2.0}


In [20]:
import torch
from sklearn.metrics import f1_score

texts = list(X_gold)
labels_true = torch.tensor(y_gold, dtype=torch.int)

# Préparer les inputs
inputs = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
inputs = {k: v.to(model_final.device) for k,v in inputs.items()}

# Prédictions
with torch.no_grad():
    logits = model_final(**inputs).logits
probs = torch.sigmoid(logits).cpu()

# Appliquer les seuils pour chaque label
thresholds = [0.5, 0.5, 0.5]
preds = (probs > torch.tensor(thresholds)).int()

In [21]:
from sklearn.metrics import classification_report

print(classification_report(labels_true, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.61      0.88      0.72        41
service livraison       0.45      0.81      0.58        21
   service client       1.00      0.07      0.14        27

        micro avg       0.56      0.62      0.59        89
        macro avg       0.69      0.59      0.48        89
     weighted avg       0.69      0.62      0.51        89
      samples avg       0.55      0.48      0.50        89



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-define

In [22]:
from sklearn.metrics import confusion_matrix

for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(labels_true[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,36,23
Vrai 1,5,36


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,58,21
Vrai 1,4,17


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,73,0
Vrai 1,25,2


## Entraînement avec dataset de validation

#### Entraînement

In [6]:
from datasets import Dataset

dataset = Dataset.from_pandas(
    df_avis[[TEXT_COL] + LABEL_COLS],
    preserve_index=True
)

dataset = dataset.train_test_split(test_size=0.20, seed=42)
train_ds = dataset["train"]
test_ds = dataset["test"]

dataset = train_ds.train_test_split(test_size=0.15, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

In [7]:
from transformers import AutoTokenizer

model_name = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map: 100%|████████████████████████████████████████████████████████████████| 1307/1307 [00:00<00:00, 10549.16 examples/s]


In [8]:
import numpy as np

def pack_labels(batch):
    import numpy as np
    batch["labels"] = np.stack([batch[col] for col in LABEL_COLS], axis=1).astype(np.float32)
    return batch

train_ds = train_ds.map(pack_labels, batched=True)
test_ds = test_ds.map(pack_labels, batched=True)
val_ds = val_ds.map(pack_labels, batched=True)

Map: 100%|█████████████████████████████████████████████████████████████████| 785/785 [00:00<00:00, 171870.79 examples/s]


In [9]:
cols_to_remove = LABEL_COLS + [TEXT_COL]
train_ds = train_ds.remove_columns(cols_to_remove)
val_ds = val_ds.remove_columns(cols_to_remove)

# on garde le texte pour l'analyse des erreurs
test_ds_for_errors = test_ds
test_ds = test_ds.remove_columns(LABEL_COLS + [TEXT_COL])

train_ds.set_format("torch")
test_ds.set_format("torch")
val_ds.set_format("torch")

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CamembertForSequenceClassification(
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./camembert_multilabel",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_4735/2251687701.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.493400,0.469192,0.702153,0.598552
2,0.402400,0.440030,0.740571,0.704580
3,0.362700,0.440312,0.743707,0.698758


TrainOutput(global_step=834, training_loss=0.4360853487924992, metrics={'train_runtime': 525.7474, 'train_samples_per_second': 25.352, 'train_steps_per_second': 1.586, 'total_flos': 1753519372448256.0, 'train_loss': 0.4360853487924992, 'epoch': 3.0})

In [138]:
trainer_final.save_model("./../models/camembert/camembert_final")
tokenizer.save_pretrained("./../models/camembert/camembert_final")

('./../models/camembert/camembert_final/tokenizer_config.json',
 './../models/camembert/camembert_final/special_tokens_map.json',
 './../models/camembert/camembert_final/sentencepiece.bpe.model',
 './../models/camembert/camembert_final/added_tokens.json',
 './../models/camembert/camembert_final/tokenizer.json')

#### Prédiction

In [14]:
def predict_texts(texts, threshold=0.5):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # déplacer les inputs sur le même device que le modèle
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits)
    preds = (probs > threshold).int()

    return probs.cpu().numpy(), preds.cpu().numpy()


In [15]:
texts_test = ["Livraison lente mais service client correct"]

probs, preds = predict_texts(texts_test)

for i, label in enumerate(label_names):
    print(label, probs[0][i], preds[0][i])

qualité produit 0.10522703 0
service livraison 0.8933151 1
service client 0.15797843 0


#### Trouver le bon seuil

In [16]:
import torch
import numpy as np

# Mettre le modèle en mode évaluation
model.eval()

val_labels = []
val_probs = []

device = next(model.parameters()).device

for batch in val_ds:
    input_ids = batch["input_ids"].unsqueeze(0).to(device)
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    probs = torch.sigmoid(logits)

    val_labels.append(labels.cpu().numpy())
    val_probs.append(probs.cpu().numpy())

# concaténer toutes les batches
val_labels = np.concatenate(val_labels, axis=0)
val_probs = np.concatenate(val_probs, axis=0)

print(val_labels.shape, val_probs.shape)

(785, 3) (785, 3)


In [17]:
from sklearn.metrics import f1_score

num_labels = val_labels.shape[1]
best_thresholds = []

for i in range(num_labels):
    best_f1 = 0
    best_t = 0.0
    for t in np.arange(0.1, 0.91, 0.01):
        y_pred = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_labels[:, i], y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    best_thresholds.append(best_t)

print("Meilleurs seuils par label :", best_thresholds)

Meilleurs seuils par label : [np.float64(0.33999999999999986), np.float64(0.44999999999999984), np.float64(0.3599999999999999)]


#### Evaluation sur le dataset de test

In [18]:
import torch
import numpy as np

# Mettre le modèle en mode évaluation
model.eval()

test_labels = []
test_probs = []

device = next(model.parameters()).device

for batch in test_ds:
    # déplacer les tenseurs sur le même device que le modèle
    input_ids = batch["input_ids"].unsqueeze(0).to(device)  # ajouter batch dimension si nécessaire
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    probs = torch.sigmoid(logits)

    test_labels.append(labels.cpu().numpy())
    test_probs.append(probs.cpu().numpy())

# concaténer tous les batches
test_labels = np.concatenate(test_labels, axis=0)
test_probs = np.concatenate(test_probs, axis=0)

print(test_labels.shape, test_probs.shape)

(1307, 3) (1307, 3)


In [19]:
preds = np.zeros_like(test_probs, dtype=int)
for i, t in enumerate(best_thresholds):
    preds[:, i] = (test_probs[:, i] > t).astype(int)

In [20]:
from sklearn.metrics import classification_report

print(classification_report(test_labels, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.81      0.84      0.83       749
service livraison       0.73      0.81      0.77       549
   service client       0.47      0.58      0.52       185

        micro avg       0.73      0.80      0.76      1483
        macro avg       0.67      0.74      0.70      1483
     weighted avg       0.74      0.80      0.77      1483
      samples avg       0.78      0.82      0.78      1483



In [21]:
from sklearn.metrics import confusion_matrix

for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(test_labels[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,413,145
Vrai 1,121,628


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,596,162
Vrai 1,105,444


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,1003,119
Vrai 1,78,107


#### Evaluation sur le dataset gold annoté manuellement

In [27]:
gold_ds = Dataset.from_pandas(
    df_avis_gold[[TEXT_COL] + LABEL_COLS],
    preserve_index=True
)

gold_ds = gold_ds.map(tokenize, batched=True)
gold_ds = gold_ds.map(pack_labels, batched=True)

# on garde le texte pour l'analyse des erreurs
gold_ds_for_errors = test_ds
gold_ds = gold_ds.remove_columns(LABEL_COLS + [TEXT_COL])
gold_ds.set_format("torch")

Map: 100%|██████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 32592.31 examples/s]


In [28]:
import torch
import numpy as np

# Mettre le modèle en mode évaluation
model.eval()

gold_labels = []
gold_probs = []

device = next(model.parameters()).device

for batch in gold_ds:
    # déplacer les tenseurs sur le même device que le modèle
    input_ids = batch["input_ids"].unsqueeze(0).to(device)  # ajouter batch dimension si nécessaire
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    probs = torch.sigmoid(logits)

    gold_labels.append(labels.cpu().numpy())
    gold_probs.append(probs.cpu().numpy())

# concaténer tous les batches
gold_labels = np.concatenate(gold_labels, axis=0)
gold_probs = np.concatenate(gold_probs, axis=0)

print(gold_labels.shape, gold_probs.shape)

(100, 3) (100, 3)


In [29]:
preds = np.zeros_like(gold_probs, dtype=int)
for i, t in enumerate(best_thresholds):
    preds[:, i] = (gold_probs[:, i] > t).astype(int)

In [30]:
from sklearn.metrics import classification_report

print(classification_report(gold_labels, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.58      0.93      0.71        41
service livraison       0.37      0.76      0.50        21
   service client       0.40      0.15      0.22        27

        micro avg       0.49      0.65      0.56        89
        macro avg       0.45      0.61      0.48        89
     weighted avg       0.47      0.65      0.51        89
      samples avg       0.53      0.51      0.50        89



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [31]:
from sklearn.metrics import confusion_matrix

for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(gold_labels[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,31,28
Vrai 1,3,38


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,52,27
Vrai 1,5,16


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,67,6
Vrai 1,23,4
